In [4]:
!pip install youtube-comment-downloader pandas scikit-learn seaborn matplotlib python-dotenv google-generativeai sentence-transformers
!pip install torch transformers accelerate bitsandbytes langchain-huggingface

In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_RECENT
import pandas as pd

def scrape_youtube_comments(video_url, max_comments=300):
    print(f"กำลังดึงคอมเมนต์จาก: {video_url}")
    downloader = YoutubeCommentDownloader()
    generator = downloader.get_comments_from_url(video_url, sort_by=SORT_BY_RECENT)
    
    comments = []
    for count, comment in enumerate(generator):
        if count >= max_comments: break
        comments.append({
            'Author': comment['author'],
            'Comment Text': comment['text']
        })
    
    df = pd.DataFrame(comments)
    df.to_csv('youtube_comments.csv', index=False, encoding='utf-8-sig')
    print(f"✅ บันทึก {len(df)} คอมเมนต์เรียบร้อย!")
    return df

# ใส่ URL วิดีโอที่ต้องการ
url = 'https://youtu.be/iogcY_4xGjo?si=yX71k-5LL2baunRC'
df = scrape_youtube_comments(url)
df.head()

กำลังดึงคอมเมนต์จาก: https://youtu.be/iogcY_4xGjo?si=yX71k-5LL2baunRC


In [ ]:
import google.generativeai as genai
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os
import pickle

# Load API Key
load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")
    
os.environ["GOOGLE_API_KEY"] = api_key
genai.configure(api_key=api_key)

# ชื่อไฟล์ Cache
CACHE_FILE = 'youtube_embeddings.pkl'

def get_embeddings(texts):
    model = 'models/text-embedding-004'
    # ยิงทีละ 100 ข้อความเพื่อป้องกัน Error
    batch_size = 100
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        result = genai.embed_content(model=model, content=batch, task_type="clustering")
        all_embeddings.extend(result['embedding'])
        
    return np.array(all_embeddings)

# เช็ค Cache
if os.path.exists(CACHE_FILE):
    print(f"✅ พบไฟล์ Cache: {CACHE_FILE} กำลังโหลด...")
    with open(CACHE_FILE, 'rb') as f:
        embeddings = pickle.load(f)
else:
    print("🚀 ไม่พบ Cache.. กำลังยิง API สร้าง Embeddings ใหม่...")
    df = pd.read_csv('youtube_comments.csv').dropna()
    embeddings = get_embeddings(df['Comment Text'].tolist())
    
    # บันทึก Cache
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(embeddings, f)
    print("💾 บันทึก Cache เรียบร้อย!")

print(f"ได้ Embeddings ขนาด: {embeddings.shape}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# โหลดข้อมูล
df = pd.read_csv('youtube_comments.csv').dropna()

# Clustering
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(embeddings)

# PCA Visualization
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)
df['pca_1'] = reduced[:, 0]
df['pca_2'] = reduced[:, 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='pca_1', y='pca_2', hue='Cluster', palette='viridis', s=50)
plt.title('Customer Segmentation (PCA)')
plt.show()

NameError: name 'pd' is not defined

In [ ]:
import sys
import os
sys.path.append('../src') # ให้ Python หาไฟล์ local_llm เจอ

from local_llm import load_local_chat_model

# 1. โหลดโมเดล
print("🔄 กำลังโหลด Local Model (Qwen)...")
local_model = load_local_chat_model()
print("✅ Local Model พร้อม!")

def dataframe_to_markdown(df):
    markdown = f"| {' | '.join(df.columns)} |\\n"
    markdown += f"| {' | '.join(['---' for _ in df.columns])} |\\n"
    for _, row in df.iterrows():
        markdown += f"| {' | '.join(str(cell).replace('\\n', ' ') for cell in row)} |\\n"
    return markdown

def analyze_persona(segment_df):
    # สุ่มตัวอย่างคอมเมนต์ 15-20 อัน
    sample = segment_df[['Author', 'Comment Text']].sample(min(20, len(segment_df)))
    table = dataframe_to_markdown(sample)
    
    prompt = f"""
    คุณคือนักวิเคราะห์ข้อมูลลูกค้า วิเคราะห์กลุ่มคอมเมนต์นี้และสร้าง "Customer Persona":
    
    ข้อมูลคอมเมนต์:
    {table}
    
    โปรดสรุปเป็นภาษาไทย:
    1. ชื่อกลุ่ม (Persona Name)
    2. พฤติกรรม/ความสนใจหลัก
    3. อารมณ์โดยรวม (Sentiment)
    """
    
    # ส่งเข้า Qwen
    res = local_model.invoke(prompt)
    return res.content

# 2. วนลูปวิเคราะห์
clusters = sorted(df['Cluster'].unique())
for c in clusters:
    print(f"\\n{'='*40}\\nวิเคราะห์กลุ่มที่ {c}\\n{'='*40}")
    segment = df[df['Cluster'] == c]
    result = analyze_persona(segment)
    print(result)